#### **_this notebook is designed to be read after notebooks related to data collection and data modelling_**

<style>
h1 {
    background-color: #CADCFC;
    color: #00246B;
    font-weight: bold;
    text-align: center;
}
</style>
<h1>
    Connecting to
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/9/93/MongoDB_Logo.svg/1200px-MongoDB_Logo.svg.png" style="height: 1em">
</h1>

In [79]:
from pymongo import MongoClient
import pandas as pd
import os
import calendar

connection_string = open(os.path.join("..", "mongo_db_connection_string.txt"), "r").read()
client = MongoClient(connection_string)
db = client["WeatherDB"]
collection_general_data = db["WeatherDB"]

#### Query 1: calculating annual/seasonal/monthly rain data (in mm) for all 8 cities in 2024:

In [147]:
pipeline = [
    {
        "$match": {
            "$expr": { 
                "$eq": [ { "$year": "$Timestamp" }, 2024 ] }
        }
    },
    {
        "$facet": {
# Annual
            "annual_rainfall": [
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": { "$year": "$Timestamp" }
                        },
                        "TotalRain": {
                            "$sum": { "$ifNull": ["$Rain.Rain_1h_mm", 0] }
                        }
                    }
                },
                { "$sort": { "_id.Location": 1, "_id.Year": 1 } }
            ],
# Seasonal
            "seasonal_rainfall": [
                {
                    "$addFields": {
                        "Month": { "$month": "$Timestamp" }
                    }
                },
                {
                    "$addFields": {
                        "Season": {
                            "$switch": {
                                "branches": [
                                    {
                                        "case": { "$in": [ "$Month", [12, 1, 2] ] },
                                        "then": "Winter"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [3, 4, 5] ] },
                                        "then": "Spring"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [6, 7, 8] ] },
                                        "then": "Summer"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [9, 10, 11] ] },
                                        "then": "Autumn"
                                    }
                                ],
                                "default": "Unknown"
                            }
                        }
                    }
                },
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": { "$year": "$Timestamp" },
                            "Season": "$Season"
                        },
                        "TotalRain": {
                            "$sum": { "$ifNull": ["$Rain.Rain_1h_mm", 0] }
                        }
                    }
                },
                {
                    "$sort": {
                        "_id.Location": 1,
                        "_id.Year": 1,
                        "_id.Season": 1
                    }
                }
            ],
# Monthly
            "monthly_rainfall": [
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": { "$year": "$Timestamp" },
                            "Month": { "$month": "$Timestamp" }
                        },
                        "TotalRain": {
                            "$sum": { "$ifNull": ["$Rain.Rain_1h_mm", 0] }
                        }
                    }
                },
                {
                    "$sort": {
                        "_id.Location": 1,
                        "_id.Year": 1,
                        "_id.Month": 1
                    }
                }
            ]
        }
    }
]

results = list(collection_general_data.aggregate(pipeline))
results

final_doc = results[0] 

def expand_facet(facet_list):
    """ 
    This function converts the facet results (a list of docs with '_id' + other fields) 
    into a Pandas DataFrame, expanding '_id' dict into columns. 
    """
    df = pd.DataFrame(facet_list)
    df = df.join(pd.DataFrame(df.pop("_id").tolist()))
    return df.drop(columns = "Year")

# Annual
df_annual = expand_facet(final_doc.get("annual_rainfall", []))
# Seasonal
df_seasonal = expand_facet(final_doc.get("seasonal_rainfall", []))
# Monthly
df_monthly = expand_facet(final_doc.get("monthly_rainfall", []))
df_monthly['Month'] = df_monthly['Month'].apply(lambda x: calendar.month_name[x])

df_annual = df_annual.rename(columns = {"TotalRain": "Year"})
locations = df_annual.pop("Location")
df_annual.insert(0, "Location", locations)
df_seasonal = df_seasonal.pivot(index = "Location", columns = "Season", values = "TotalRain").reset_index()
df_monthly = df_monthly.pivot(index = "Location", columns = "Month", values = "TotalRain").reset_index()

merged_df = pd.merge(pd.merge(df_annual, df_seasonal), df_monthly)
month_names = [calendar.month_name[i] for i in range(1, 13)]
merged_df = merged_df.reindex(columns = ['Location', 'Year', 'Winter', 'Spring', 'Summer', 'Autumn'] + month_names)

def highlight_average(row):
    if row["Location"] == "Average":
        return ["font-weight: bold"] * len(row)
    return [""] * len(row)

def display_with_average(df, index_column, precision = 2, non_index_labels = []):
    indices = df.pop(index_column)
    df.insert(0, index_column, indices)
    average_row = df.drop(columns = [index_column] + non_index_labels).mean().to_frame().T
    average_row.insert(0, index_column, "Average") 
    df_with_average = pd.concat([df, average_row], ignore_index = True)
    display(df_with_average.style.format(precision = precision).apply(highlight_average, axis = 1).hide(axis = "index"))

print("Total rainfall (mm) in location in time periods:")
display_with_average(merged_df, "Location")

Total rainfall (mm) in location in time periods:


Location,Year,Winter,Spring,Summer,Autumn,January,February,March,April,May,June,July,August,September,October,November,December
Buckinghamshire,1491.68,448.78,358.44,192.08,492.38,122.90,262.38,127.52,154.32,76.60,37.56,110.46,44.06,288.00,89.96,114.42,63.50
East Sussex,1957.44,498.36,489.26,283.56,686.26,132.30,290.96,165.86,147.32,176.08,39.22,159.34,85.00,342.92,222.38,120.96,75.10
Essex,1738.46,504.34,461.42,328.78,443.92,139.58,238.22,142.26,177.68,141.48,22.76,262.68,43.34,226.32,93.78,123.82,126.54
Hertfordshire,2043.78,613.82,601.48,257.76,570.72,213.76,286.26,191.08,188.56,221.84,24.22,187.20,46.34,322.44,121.84,126.44,113.80
Kent,1701.86,489.16,411.20,266.62,534.88,134.36,252.62,166.20,131.30,113.70,86.10,109.92,70.60,231.00,158.02,145.86,102.18
London,1935.38,475.86,581.12,310.04,568.36,143.88,250.92,196.08,150.52,234.52,38.54,212.66,58.84,230.90,141.98,195.48,81.06
Surrey,1266.78,318.82,377.96,112.30,457.70,91.90,168.00,157.82,91.72,128.42,16.14,65.04,31.12,201.38,130.86,125.46,58.92
West Sussex,1869.10,481.74,516.06,231.28,640.02,115.82,276.80,220.10,113.66,182.30,42.84,126.90,61.54,284.52,220.18,135.32,89.12
Average,1750.56,478.86,474.62,247.80,549.28,136.81,253.27,170.87,144.38,159.37,38.42,154.28,55.11,265.94,147.38,135.97,88.78


#### Query 2: Calculating the annual **Average** air pollution data for all cities in 2024

In [148]:
pipeline = [
    {
        "$match": {
            "$expr": {
                "$eq": [
                    { "$year": "$Timestamp" }, 
                    2024
                ]
            }
        }
    },
    {
        "$group": {
            "_id": {
                "Location": "$Location",
                "Year": { "$year": "$Timestamp" }
            },
            "AQI":  { "$avg": "$Air_Pollution.AQI" },
            "CO":   { "$avg": "$Air_Pollution.CO" },
            "NO":   { "$avg": "$Air_Pollution.NO" },
            "NO2":  { "$avg": "$Air_Pollution.NO2" },
            "O3":   { "$avg": "$Air_Pollution.O3" },
            "SO2":  { "$avg": "$Air_Pollution.SO2" },
            
# Firstly, I need to rename PM2.5 as pm2_5 becuase if I use ".", error occurs.
# Secondly, I think there is a fundamental error with MongoDB it self, I cannot get what I want just by typing pm2\.5, I need to use $getField to get the value of PM2.5.

            "PM2_5": {
                "$avg": {
                    "$getField": {
                        "field": "PM2.5",
                        "input": "$Air_Pollution"
                    }
                }
            },
            "PM10": { "$avg": "$Air_Pollution.PM10" },
            "NH3":  { "$avg": "$Air_Pollution.NH3" }
        }
    },
    {
        "$project": {
            "_id": 0,
            "Location": "$_id.Location",
            "Year": "$_id.Year",
            "AQI": 1,
            "CO": 1,
            "NO": 1,
            "NO2": 1,
            "O3": 1,
            "SO2": 1,
            "PM2_5": 1,
            "PM10": 1,
            "NH3": 1
        }
    },
    {
        "$sort": {
            "Location": 1,
            "Year": 1
        }
    }
]

results = list(collection_general_data.aggregate(pipeline))

df = pd.DataFrame(results).drop(columns="Year")

df.rename(columns={
    "O3": "O3 (Crucial/Most Harmful)",
    "PM2_5": "PM2.5 (Crucial/Most Harmful)"
}, inplace=True)

display_with_average(df, "Location", 5)

Location,AQI,CO,NO,NO2,O3 (Crucial/Most Harmful),SO2,PM2.5 (Crucial/Most Harmful),PM10,NH3
Buckinghamshire,1.68262,221.72382,0.25923,3.95641,61.09361,1.08978,3.81290,3.80049,1.95040
East Sussex,1.87476,227.37688,0.51023,6.18409,69.20285,2.59569,4.86573,5.34998,0.79891
Essex,1.70583,232.46629,1.42064,9.33633,58.46953,3.10123,4.59944,4.82291,1.04413
Hertfordshire,1.67429,231.28632,1.36484,9.79511,56.90881,3.09821,4.37097,4.58870,0.92469
Kent,1.73893,225.82381,0.75194,6.02101,62.04775,1.96739,4.63156,4.99522,0.98865
London,1.68821,250.90783,5.76573,17.29157,50.92735,7.52799,5.37116,5.63128,0.82823
Surrey,1.71238,231.41025,2.05732,9.96325,58.57696,3.22558,4.30370,4.41637,0.83049
West Sussex,1.75000,223.58687,0.49480,6.39981,62.90793,2.21789,4.44628,4.75808,1.06806
Average,1.72838,230.57276,1.57809,8.61845,60.01685,3.10297,4.55022,4.79538,1.05420


#### Query 3: get the **Average** solar irradiance data for all cities in 2024

In [149]:
pipeline = [
    {
        "$match": {
            "$expr": {
                "$eq": [
                    {"$year": "$Timestamp"}, 
                    2024
                ]
            }
        }
    },
    {
        "$group": {
            "_id": {
                "Location": "$Location",
                "Year": {"$year": "$Timestamp"}
            },
            "avgDNI": { "$avg": "$Solar_Irradiation.dni" },
            "avgGHI": { "$avg": "$Solar_Irradiation.ghi" }
        }
    },
    {
        "$project": {
            "_id": 0,
            "Location": "$_id.Location",
            "Year": "$_id.Year",
            "avgDNI": 1,
            "avgGHI": 1
        }
    },
    {
        "$sort": {
            "Location": 1,
            "Year": 1
        }
    }
]


results = list(collection_general_data.aggregate(pipeline))

df = pd.DataFrame(results).drop(columns = "Year")

df.rename(columns = {"avgDNI": "DNI", "avgGHI": "GHI"}, inplace = True)

display_with_average(df, "Location", 6)

Location,DNI,GHI
Buckinghamshire,111.526071,125.099762
East Sussex,128.789286,135.524881
Essex,110.396786,127.948095
Hertfordshire,109.494167,125.339048
Kent,111.990238,128.328452
London,105.733690,121.177024
Surrey,103.352738,123.144405
West Sussex,123.433095,132.635476
Average,113.089509,127.399643


#### Query 4: Get the total snowing days (if the value of Snow_1h_mm is non-zero in any day, then that day is a snowing day) for all cities in 2024

In [150]:
pipeline = [
    {
        "$match": {
            "$expr": {
                "$eq": [
                    { "$year": "$Timestamp" },
                    2024
                ]
            }
        }
    },
# Group by (Location, Year, Month, Day), and mark if any hour had Snow_1h_mm > 0

    {
        "$group": {
            "_id": {
                "Location": "$Location",
                "Year": { "$year": "$Timestamp" },
                "Month": { "$month": "$Timestamp" },
                "Day": { "$dayOfMonth": "$Timestamp" }
            },
            "snowIndicator": {
                "$max": {
                    "$cond": [
                        { "$gt": [ "$Snow.Snow_1h_mm", 0 ] },
                        1,
                        0
                    ]
                }
            }
        }
    },
# Group by (Location, Year) summing daily indicators => total snow days

    {
        "$group": {
            "_id": {
                "Location": "$_id.Location",
                "Year": "$_id.Year"
            },
            "SnowDays": { "$sum": "$snowIndicator" }
        }
    },

    {
        "$project": {
            "_id": 0,
            "Location": "$_id.Location",
            "Year": "$_id.Year",
            "SnowDays": 1
        }
    },
    {
        "$sort": {
            "Location": 1,
            "Year": 1
        }
    }
]

results = list(collection_general_data.aggregate(pipeline))

df = pd.DataFrame(results).drop(columns = "Year")

display_with_average(df, "Location", 0)

Location,SnowDays
Buckinghamshire,7
East Sussex,2
Essex,6
Hertfordshire,4
Kent,3
London,3
Surrey,3
West Sussex,4
Average,4


#### Query 5: Calculate annual/seasonal/monthly **Average day's Max/Min Temperature** (ºC) for all 8 cities in 2024:  
##### (in MongoDB database, temperature is stored in Kelvin, so we need to do some modifications: **subtract raw data by 273.15 to get temperature in degree celsius**. We chose celsius units to make the results more intuitive to most people)

In [154]:
pipeline = [
    {
        "$match": {
            "$expr": {
                "$eq": [
                    {"$year": "$Timestamp"},
                    2024
                ]
            }
        }
    },

    {
        "$facet": {
            "annual_temps": [
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": {"$year": "$Timestamp"}
                        },
# Convert from Kelvin to Celsius for Temp_Max and Temp_Min
                        "AvgTempMaxC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Max",
                                    273.15
                                ]
                            }
                        },
                        "AvgTempMinC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Min",
                                    273.15
                                ]
                            }
                        }
                    }
                },
                {
                    "$project": {
                        "_id": 0,
                        "Location": "$_id.Location",
                        "Year": "$_id.Year",
                        "AvgTempMaxC": 1,
                        "AvgTempMinC": 1
                    }
                },
                {
                    "$sort": {
                        "Location": 1,
                        "Year": 1
                    }
                }
            ],
            "seasonal_temps": [
                {
                    "$addFields": {
                        "Month": { "$month": "$Timestamp" }
                    }
                },
                {
                    "$addFields": {
                        "Season": {
                            "$switch": {
                                "branches": [
                                    {
                                        "case": { "$in": [ "$Month", [12, 1, 2] ] },
                                        "then": "Winter"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [3, 4, 5] ] },
                                        "then": "Spring"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [6, 7, 8] ] },
                                        "then": "Summer"
                                    },
                                    {
                                        "case": { "$in": [ "$Month", [9, 10, 11] ] },
                                        "then": "Autumn"
                                    }
                                ],
                                "default": "Unknown"
                            }
                        }
                    }
                },
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": { "$year": "$Timestamp" },
                            "Season": "$Season"
                        },
                        "AvgTempMaxC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Max",
                                    273.15
                                ]
                            }
                        },
                        "AvgTempMinC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Min",
                                    273.15
                                ]
                            }
                        }
                    }
                },
                {
                    "$project": {
                        "_id": 0,
                        "Location": "$_id.Location",
                        "Year": "$_id.Year",
                        "Season": "$_id.Season",
                        "AvgTempMaxC": 1,
                        "AvgTempMinC": 1
                    }
                },
                {
                    "$sort": {
                        "Location": 1,
                        "Year": 1,
                        "Season": 1
                    }
                }
            ],
            "monthly_temps": [
                {
                    "$group": {
                        "_id": {
                            "Location": "$Location",
                            "Year": { "$year": "$Timestamp" },
                            "Month": { "$month": "$Timestamp" }
                        },
                        "AvgTempMaxC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Max",
                                    273.15
                                ]
                            }
                        },
                        "AvgTempMinC": {
                            "$avg": {
                                "$subtract": [
                                    "$Temperature.Temp_Min",
                                    273.15
                                ]
                            }
                        }
                    }
                },
                {
                    "$project": {
                        "_id": 0,
                        "Location": "$_id.Location",
                        "Year": "$_id.Year",
                        "Month": "$_id.Month",
                        "AvgTempMaxC": 1,
                        "AvgTempMinC": 1
                    }
                },
                {
                    "$sort": {
                        "Location": 1,
                        "Year": 1,
                        "Month": 1
                    }
                }
            ]
        }
    }
]

results = list(collection_general_data.aggregate(pipeline))

final_doc = results[0]

df_annual = pd.DataFrame(final_doc["annual_temps"]).drop(columns = "Year")

df_seasonal = pd.DataFrame(final_doc["seasonal_temps"]).drop(columns = "Year")

df_monthly = pd.DataFrame(final_doc["monthly_temps"]).drop(columns = "Year")
df_monthly['Month'] = df_monthly['Month'].apply(lambda x: calendar.month_name[x])

print("\n=== Annual Temperature Data (°C) ===")
display_with_average(df_annual, "Location", 6)

print("\n=== Seasonal Temperature Data (°C) ===")
display_with_average(df_seasonal, "Location", 6, ["Season"])

print("\n=== Monthly Temperature Data (°C) ===")
display_with_average(df_monthly, "Location", 6, ["Month"])


=== Annual Temperature Data (°C) ===


Location,AvgTempMaxC,AvgTempMinC
Buckinghamshire,12.384239,10.232150
East Sussex,13.334062,10.823354
Essex,13.212248,10.657167
Hertfordshire,13.060337,10.536615
Kent,13.591689,10.851283
London,13.504560,10.822075
Surrey,13.361983,10.651054
West Sussex,13.285058,10.569517
Average,13.216772,10.642902



=== Seasonal Temperature Data (°C) ===


Location,AvgTempMaxC,AvgTempMinC,Season
Buckinghamshire,11.952001,9.970250,Autumn
Buckinghamshire,11.713909,9.364380,Spring
Buckinghamshire,17.664102,15.370519,Summer
Buckinghamshire,7.583807,5.631297,Winter
East Sussex,13.224460,11.042659,Autumn
East Sussex,12.430254,9.662763,Spring
East Sussex,18.499139,15.780782,Summer
East Sussex,8.569338,6.221676,Winter
Essex,12.822821,10.466193,Autumn
Essex,12.402939,9.694787,Spring



=== Monthly Temperature Data (°C) ===


Location,AvgTempMaxC,AvgTempMinC,Month
Buckinghamshire,5.909681,3.813472,January
Buckinghamshire,8.996437,6.905977,February
Buckinghamshire,9.197218,7.036532,March
Buckinghamshire,10.727847,8.510431,April
Buckinghamshire,15.184852,12.518629,May
Buckinghamshire,16.176569,13.593069,June
Buckinghamshire,18.002141,15.862328,July
Buckinghamshire,18.787419,16.630551,August
Buckinghamshire,15.250187,13.088807,September
Buckinghamshire,12.358239,10.465094,October
